# KanoFarm AI: train the Plant Doctor (starter model)
**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run each cell in order.
This trains on PlantDoc (CC BY 4.0). It covers **tomato, potato and pepper** well enough to test the pipeline. It does **not** cover cowpea, groundnut, rice, sorghum, millet, cassava or onion, and it is **not validated for Nigerian farms**.

In [ ]:
!nvidia-smi | head -5

In [ ]:
# EDIT THESE TWO LINES to match your own repo.
# REPO is the full clone URL. SUBDIR is the path INSIDE the repo where kanofarm-ai actually
# lives -- leave it as "" if your repo's top level IS the project (index.py sits at the root).
# Example matching a project nested inside a subfolder, as yours is:
#   REPO = "https://github.com/Ibrahimgama-4/Nigerian-smartAI"
#   SUBDIR = "kanofarm-ai-complete"
REPO = "https://github.com/YOUR-NAME/kanofarm-ai"
SUBDIR = ""

import os, subprocess
folder = REPO.rstrip("/").split("/")[-1]
subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
target = os.path.join(folder, SUBDIR) if SUBDIR else folder
assert os.path.isfile(os.path.join(target, "ml", "training", "train.py")), (
    f"Could not find ml/training/train.py under '{target}'. Check SUBDIR matches your repo's actual "
    f"folder layout (open the repo on GitHub and look for where ml/ and index.py actually sit).")
os.chdir(target)
print("Working directory is now:", os.getcwd())


In [ ]:
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset

In [ ]:
# Keep only the crops we can support. Add ,maize ONLY after you add your own maize___healthy images.
!python -m ml.preprocessing.prepare_plantdoc --src PlantDoc-Dataset --dst data/plantdoc --crops tomato,potato,pepper

**Read the output above.** `unmatched_folders` lists folders the script could not map. If your desired classes are missing, folder names differ and the map file needs adjusting.

In [ ]:
!python -m ml.training.build_splits --src data/plantdoc --dst data/split --presplit

**Sanity check before training.** This reuses the exact same check `train.py` uses, so if this cell says OK, training will not fail on a class mismatch. If it finds a problem, it tells you precisely which class is missing from which split -- fix that before running the Train cell below.

In [ ]:
import os
from torchvision.datasets import ImageFolder
from ml.training.class_check import describe_class_mismatch

tr = ImageFolder("data/split/train").classes
va = ImageFolder("data/split/val").classes
te = ImageFolder("data/split/test").classes
print(f"train: {len(tr)} classes | val: {len(va)} classes | test: {len(te)} classes")

problem = describe_class_mismatch("data/split", tr, va, te)
if problem:
    print("\n❌ PROBLEM FOUND:\n")
    print(problem)
else:
    print("\n✅ All three splits have the exact same classes. Safe to run the Train cell.")


In [ ]:
!python -m ml.training.train --data data/split --out ml/models/v1 \
  --dataset-name "PlantDoc (CC BY 4.0)" --dataset-version "github-2020" --epochs 15

In [ ]:
import json
card = json.load(open("ml/models/v1/model_card.json"))
m = card["metrics"]
print("test images:", m["n"], "| accuracy:", m["accuracy"], "| macro F1:", m["macro_f1"])
print("\nPer-class (worst recall first):")
for k, v in sorted(m["per_class"].items(), key=lambda kv: kv[1]["recall"]):
    print(f'{k:40s} recall={v["recall"]:.2f} precision={v["precision"]:.2f} n={v["support"]}')

**Look at the worst classes.** Low recall or tiny test counts mean the model is unreliable for them. Do not trust accuracy alone.

In [ ]:
!cp data/plantdoc/label_meta.json ml/models/v1/label_meta.json
!mkdir -p out/model && cp ml/models/v1/model.pt ml/models/v1/model_card.json ml/models/v1/label_meta.json out/model/
!cp ml/serving/hf_space/* out/ 2>/dev/null; ls -R out | head -20
!cd out && zip -r ../plant_doctor_v1.zip . >/dev/null
from google.colab import files
files.download("plant_doctor_v1.zip")